In [ ]:
import asyncio
import os

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

In [ ]:

client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

In [ ]:
# Create a Writer agent that generates content
writer_agent = client.as_agent(
    name="Writer",
    instructions=(
        "You are an excellent content writer. You create new content and edit contents based on the feedback."
    ),
)

# Create a Reviewer agent that provides feedback
reviewer_agent = client.as_agent(
    name="Reviewer",
    instructions=(
        "You are an excellent content reviewer. "
        "Provide actionable feedback to the writer about the provided content. "
        "Provide the feedback in the most concise manner possible."
    ),
)

In [ ]:
# Build the workflow with agents as executors
workflow = WorkflowBuilder(start_executor=writer_agent).add_edge(writer_agent, reviewer_agent).build()

In [ ]:
last_author: str | None = None

events = workflow.run("Inspire software engineer to pickup AI development, please provide hints", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n\n{author}: {update.text}", end="", flush=True)
            last_author = author
        else:
            print(update.text, end="", flush=True)